# Boom Level Accuracy
In theis notebok we prompt engineered a Bloom evaluator agent whose input is a question and whose output is a label remember, understand, or apply. Using a dataset of past exam questions, we refined the prompting until the evaluator agreed with our human-labeled dataset 86% of the time.

We prompt-engineered one question-generator agent to produce all three Bloom levels and generated 1000 questions for each level. We also created three separate level-specific agents one each for Remember, Understand, and Apply and generated 1000 questions from each of them.

We evaluated all generated questions using our prompt engineered Bloom evaluator.
 Single-agent accuracy: Remember 100%, Understand 100%, Apply 97.7%.
 One-agent-per-level accuracy: Remember 99.1%, Understand 100%, Apply 99.2%.




First install necessary libaires and set API key.

In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI
import ast
import csv
import pandas as pd
import time

In [ ]:
client = OpenAI(api_key="sk-proj-")

## Evaluator

Now lets create an evaultor using our anatomy dataset. <br>
First we read in anatomy class data that contains questions labeled remember, understand, apply.

In [ ]:
df = pd.read_csv("questions_self_labeled.csv", encoding="latin-1")

questions = df["question"].tolist()
labels = df["label"].tolist()

Now we create an evaluator agent.

In [ ]:
system_prompt_evaluator = """
You classify a question into exactly one Bloom level:
- remember
- understand
- apply

## DEFINITIONS

### REMEMBER
The question asks for:
- a fact, term, definition, or basic property
- direct retrieval of information without interpretation
- listing, identifying, or naming something

Typical forms:
"What is…?"
"Define…"
"List the…"
"Who/what/when…?"

### UNDERSTAND
The question asks for:
- explanation, interpretation, or conceptual reasoning
- describing relationships, meaning, or significance
- summarizing, comparing, or explaining why something occurs
- cause–effect reasoning **without** a real scenario

Typical forms:
"Explain why…"
"Describe how…"
"What is the meaning of…?"
"Compare…"

### APPLY
The question presents a **new situation or scenario** and requires using knowledge to reach a conclusion or perform a task.
Apply questions ALWAYS involve:
- a concrete situation, problem, or case
- a change, event, condition, or hypothetical setup
- using learned rules or concepts to determine an outcome, solve something, or decide what to do

Common patterns:
- “Given this situation…”
- “If X occurs, what follows?”
- “In the following case…”
- “How would you use concept Y to solve…?”
- “What should be done in this scenario?”

Apply requires **transferring knowledge to a situation that differs from the original source material**.

## DECISION RULES
Follow this order when classifying:

1. **If the question includes a scenario or a new situation requiring knowledge to be applied → APPLY.**
2. Otherwise, if it asks for explanation, interpretation, or conceptual understanding → UNDERSTAND.**
3. Otherwise, if it requires only factual recall → REMEMBER.**

## OUTPUT FORMAT
Output only one label:
"remember", "understand", or "apply".
"""

Create a function to intake question list and output labeled questions.

In [ ]:
def classify_questions(question_list):
    predicted_questions = []
    predicted_labels = []

    for q in question_list:
        time.sleep(1.1)
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "system", "content": system_prompt_evaluator},
                {"role": "user", "content": q}
            ]
        )

        label = response.choices[0].message.content.strip().lower()

        predicted_questions.append(q)
        predicted_labels.append(label)

        # print(f"Question: {q}")
        # print(f" → {label}\n")

    return predicted_questions, predicted_labels

First lets runs the classifier on our anatomy data.

In [ ]:
questions, gpt_labels= classify_questions(questions)

Function to compute accuracy between label predicted by the evaluator and true lable.

In [ ]:
def compute_accuracy(true_labels, predicted_labels):
    correct = 0
    total = len(true_labels)

    for true_label, pred_label in zip(true_labels, predicted_labels):
        if true_label.strip().lower() == pred_label.strip().lower():
            correct += 1

    accuracy = correct / total
    print(f"Accuracy: {accuracy:.2%}")

    return accuracy

Now we compute the accuracy of our self made labeling for the antonomy questions.

In [ ]:
accuracy = compute_accuracy(labels, gpt_labels)

Accuracy: 86.09%


We use slide text to generate questions. <br>
We will read in this text from a slide_info.txt.

In [ ]:
# read in data
with open("slide_info.txt", "r") as f:
    data_str = f.read()

data = ast.literal_eval(data_str)

# Extract raw_text fields
raw_text_list = [topic["raw_text"] for topic in data["topics"]]

Next we create a function that will generate questions. <br>
Input is a list of slide text and a system prompt.

In [ ]:
def generate_questions(raw_text_list, system_prompt, n=1000, filename="questions.csv"):
    examples = []

    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)

        for i in range(n):
            slide_text = raw_text_list[i % len(raw_text_list)]    # for text from one slides

            response = client.chat.completions.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": slide_text}
                ]
            )

            question = response.choices[0].message.content.strip()

            writer.writerow([question])
            examples.append(question)

            print(f"{i+1}/{n}: {question}")

    print(f"\nSaved {n} questions to {filename}")
    return examples


## 3 Agents to Produce Questiions

We create a prompt for an agent to generate remember level questions.

In [ ]:
system_prompt_remember = """
You are an agent that generates Remember-level questions.
Your input is raw text from lecture slides, but you must NOT refer to the text or say "in the slide" or "mentioned."

Your task:
Create one factual recall question based on the anatomical facts found in the text.

Rules:
- The question must have one clear, factual answer.
- Do NOT ask the learner to explain, compare, or analyze.
- The question must sound like a normal test question.
- Do NOT mention or reference “the slide,” “the text,” or “the passage.”
- Keep the question short and specific.
- Use only information that actually appears in the input text.
- If the text is sparse, pick any clear fact and ask a recall question about it.

Output:
One Remember-level question.
"""

What are the four basic tissue types in the human body?
What are the four basic tissue types in the human body?
What are the four basic tissue types in the human body?
What are the four basic tissue types in the human body?
What are the four basic tissue types in the human body?
What are the four basic tissue types in the human body?


Now generate 1000 remember questions.

In [ ]:
remember_questions = generate_questions(raw_text_list, system_prompt_remember, n=1000, filename="remember_questions.csv")

1/1000: What are the four basic types of tissue in the human body?
2/1000: What component makes up most of the volume in connective tissue?
3/1000: What type of protein fiber in connective tissue provides tensile strength?
4/1000: What size range do lymph nodes normally measure?
5/1000: From what embryonic tissue do connective tissue cells originate?
6/1000: What type of connective tissue contains parallel collagen fibers and provides tensile strength along one axis?
7/1000: What type of cell is sparsely distributed within the extracellular matrix of cartilage?
8/1000: What type of bone houses red bone marrow, the site of hematopoiesis?
9/1000: What are the four basic tissue types in the human body?
10/1000: What type of cell produces and secretes protein fibres in connective tissue?
11/1000: What is the name of the protein fibres in connective tissue that provide resiliency?
12/1000: What is the normal size range of lymph nodes in millimeters?
13/1000: From what embryonic tissue do co

Create a prompt for an agent to produce only understand questions.

In [ ]:
system_prompt_understand = """
You are an agent that generates Understand-level questions.
Your input is raw text from lecture slides, which may be short or incomplete.

Your task:
Create one question that requires the learner to demonstrate comprehension
of meaning, concepts, or relationships based on the input text.

Rules:
- The question must require the learner to explain, summarize, interpret, or describe.
- Do NOT ask the learner to apply, solve, or analyze.
- The question must sound like a normal test question.
- Do NOT mention or reference “the slide,” “the text,” or “this passage.”
- Keep the question clear and specific.
- Use only information that appears in the input text.
- If the text is sparse, choose any concept and ask for an explanation or interpretation.

Output:
One Understand-level question.
"""

Generate 1000 understand questions.

In [ ]:
understand_questions = generate_questions(raw_text_list, system_prompt_understand, n=1000, filename="understand_questions.csv")

1/1000: Explain the relationship between cells, tissues, and organs in terms of their structure and function within the human body.
2/1000: Explain how the composition and structure of the extracellular matrix (ECM) influence the properties and functions of different types of connective tissue.
3/1000: Summarize the main components of connective tissue and describe the role each plays in maintaining the tissue’s structure and function.
4/1000: Describe the role of lymph nodes in preventing the spread of disease and infection within the body.
5/1000: Explain how the type and proportion of cells, protein fibres, and ground substance contribute to the characteristics and functions of different connective tissues.
6/1000: Describe the differences between loose (areolar) connective tissue and dense connective tissue, including their structural characteristics and functions.
7/1000: Explain how the properties of cartilage’s extracellular matrix contribute to its ability to withstand compress

Create a prompt for agent to produce only apply level questions.

In [ ]:
system_prompt_apply = """
You generate Apply-level questions from raw slide text.

Your task:
Create ONE scenario-based question that requires the learner to USE information
from the text to determine an outcome, predict an effect, or make a decision
in a new situation.

Apply-level requirements:
- Include a concrete scenario, case, or event.
- Something in the scenario must change, malfunction, be damaged, vary, or occur.
- The learner must apply knowledge from the text to predict the outcome of that change.
- The question must have one clear, logically deducible answer.
- Avoid explanation-style prompts.

Do NOT:
- Ask “why,” “explain,” or “describe.”
- Ask for mechanisms or summaries.
- Refer to “the text” or “the slide.”

Preferred structures:
- “If X condition occurs, what would happen to Y?”
- “Given this situation, what outcome should be expected?”
- “After event X, what result would most likely follow?”

Output:
ONE Apply-level scenario question.
"""

In [ ]:
apply_questions = generate_questions(raw_text_list, system_prompt_apply, n=1000, filename="apply_questions.csv")

1/1000: If an injury causes widespread damage to the connective tissue in an organ while leaving the other tissue types mostly intact, what effect would this most likely have on the organ’s overall structure?
2/1000: If a genetic mutation caused connective tissue cells to significantly reduce the secretion of protein fibres into the extracellular matrix, what change would most likely occur in the properties of the tissue?
3/1000: A patient develops a mutation that significantly reduces the production of proteoglycans in their connective tissue. What would most likely happen to the tissue’s ability to resist compression?
4/1000: If cancer cells block several lymphatic vessels leading to the axillary lymph nodes, what effect would most likely be observed in the tissue of the upper limb on that side?
5/1000: If an individual experiences a genetic mutation that prevents the normal development of mesenchyme during embryogenesis, what would most likely be the result for their connective tiss

## Accuracy of 3 Seperate Agents.

Now we compute the accuracy for each of the remmeber, understand, apply agents.

First the remember agent.

In [ ]:
df = pd.read_csv("remember_questions.csv", header=None)
remember_questions = df[0].tolist()

In [ ]:
_, gpt_labels =classify_questions(remember_questions)
true_labels = ["remember"] * len(remember_questions)
accuracy = compute_accuracy(true_labels, gpt_labels)

Accuracy: 99.10%


Lets inspect the misclassificaitons.

In [ ]:
misclassified = []

for q, pred, true in zip(remember_questions, gpt_labels, true_labels):
    if pred != true:
        misclassified.append((q, pred, true))

# View results
for q, pred, true in misclassified:
    print("QUESTION:")
    print(q)
    print("Predicted:", pred)
    print("True:", true)
    print("-" * 80)


QUESTION:
What determines the physical properties of connective tissue?
Predicted: understand
True: remember
--------------------------------------------------------------------------------
QUESTION:
What determines the properties of connective tissue?
Predicted: understand
True: remember
--------------------------------------------------------------------------------
QUESTION:
What determines the properties of connective tissue?
Predicted: understand
True: remember
--------------------------------------------------------------------------------
QUESTION:
What determines the properties of connective tissue?
Predicted: understand
True: remember
--------------------------------------------------------------------------------
QUESTION:
What determines the properties of connective tissue?
Predicted: understand
True: remember
--------------------------------------------------------------------------------
QUESTION:
What determines the properties of connective tissue?
Predicted: understand
T

Next the understand questions.

In [ ]:
df = pd.read_csv("understand_questions.csv", header=None)
understand_questions = df[0].tolist()

In [ ]:
_, gpt_labels = classify_questions(understand_questions)
true_labels = ["understand"] * len(understand_questions)
accuracy = compute_accuracy(true_labels , gpt_labels)

Accuracy: 100.00%


Next the apply questions.

In [ ]:
df = pd.read_csv("apply_questions.csv", header=None)
apply_questions = df[0].tolist()

In [ ]:
_, gpt_labels = classify_questions(apply_questions)
true_labels = ["apply"] * len(apply_questions)
accuracy = compute_accuracy(true_labels, gpt_labels)

Accuracy: 99.20%


Lets insepct the apply misclassifications.

In [ ]:
misclassified = []

for q, pred, true in zip(apply_questions, gpt_labels, true_labels):
    if pred != true:
        misclassified.append((q, pred, true))

# View results
for q, pred, true in misclassified:
    print("QUESTION:")
    print(q)
    print("Predicted:", pred)
    print("True:", true)
    print("-" * 80)

QUESTION:
If a person's articular cartilage is damaged due to trauma, what effect would this have on the tissue's ability to repair itself?
Predicted: understand
True: apply
--------------------------------------------------------------------------------
QUESTION:
If a mutation prevents certain cells from differentiating properly during development, what effect would this most likely have on the structure and function of the tissues and organs formed from those cells?
Predicted: understand
True: apply
--------------------------------------------------------------------------------
QUESTION:
If a heavy trauma causes damage to the cartilage in a joint, what would most likely be the expected outcome in terms of repair speed and tissue regeneration?
Predicted: understand
True: apply
--------------------------------------------------------------------------------
QUESTION:
If an injury causes significant damage to the extracellular matrix of articular cartilage in a joint, what is the most 

## Single Agent to Produce 3 Level of Questions

Now lets try one central agent that produces remember, understand, apply questions

In [ ]:
system_prompt_three_level = """
You are an agent that generates questions at a specified Bloom’s Taxonomy level.
You will receive two inputs:
1. Raw text extracted from lecture slides.
2. A level request: "remember", "understand", or "apply".

Your task:
Create ONE question at the requested level based only on the information in the slide text.

Rules by level:

remember:
- Ask for simple factual recall only.
- The question must require recalling a fact, term, definition, list, or identification.
- One clear factual answer.
- No explanation, interpretation, mechanism, reasoning, or relationships.
- Do NOT ask what determines, causes, influences, controls, affects, or results in anything.
- Do NOT ask about functions or purposes unless they are explicitly stated facts.
- Do NOT ask about properties unless the property itself is a memorized fact.
- No scenario, no prediction, no cause–effect language.

understand:
- Ask the learner to explain, describe, summarize, or interpret a concept.
- No scenario-based application or problem-solving.
- The question should test comprehension of meaning, relationships, or mechanisms.
- Do NOT ask the learner to predict outcomes in new situations.

apply:
- Create a NEW scenario, event, or condition that is directly relevant to the information.
- The scenario must include a change, malfunction, variation, or specific situation the learner must reason about.
- The learner must USE information from the text to PREDICT an outcome, determine a result, or identify the consequence of that change.
- The answer must be a single, logically deducible outcome.
- The question must be an ACTUAL question ending with a question mark.
- Do NOT reveal, imply, or restate the outcome inside the question.
- Do NOT write the question as a statement or give away the result (e.g., “If X happens, the tissue would…”).
- Do NOT ask for interpretation of signs, meaning, or function (“what does this indicate,” “what does this mean”).
- Do NOT use explanation-style wording (“why,” “explain,” “describe”).
- Do NOT ask the learner to restate normal function; require a predicted outcome of the scenario.

General rules:
- Do NOT reference “the slide” or “the text.”
- Use only information found in the input text.
- Keep the question clear, concise, and natural.
- If the slide text is sparse, choose any fact present and build the question around it.
- The output must ALWAYS be a question.

Output:
Only the question.

"""

Create a function that will generate questions at three levels using the above prompt.

In [ ]:
def generate_all_levels(raw_text_list, iterations=10, filename="three_level_agent_questions_labels.csv"):
    levels = ["remember", "understand", "apply"]
    all_questions = []
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["question", "label"])  # header

        for level in levels:

            for i in range(iterations):
                slide_text = raw_text_list[i % len(raw_text_list)]
                time.sleep(1.1)
                user_prompt = f"""
                  Slide text:
                  {slide_text}
                  Requested level: {level}
                  """

                response = client.chat.completions.create(
                    model="gpt-4.1",
                    messages=[
                        {"role": "system", "content": system_prompt_three_level},
                        {"role": "user", "content": user_prompt}
                    ]
                )

                question = response.choices[0].message.content.strip()

                writer.writerow([question, level])  # save immediately

                all_questions.append((question, level))

                print(f"[{level}] {i+1}/{iterations}: {question}")

        return all_questions

Generate 1000 questions for each level.

In [ ]:
all_questions = generate_all_levels(raw_text_list, iterations=300, filename="three_level_agent_questions_labels.csv")

[remember] 1/1000: What are the four basic tissue types?
[remember] 2/1000: What is the main component of connective tissue by volume?
[remember] 3/1000: What is the term for an increase in tissue fluid volume?
[remember] 4/1000: What is the normal size range of lymph nodes?
[remember] 5/1000: From what embryonic tissue do connective tissue cells originate?
[remember] 6/1000: What type of connective tissue has parallel collagen fibres providing tensile strength along one axis?
[remember] 7/1000: What are the three subtypes of cartilage?
[remember] 8/1000: What type of bone contains spaces that house red bone marrow?
[remember] 9/1000: What are the four basic tissue types?
[remember] 10/1000: What is the main component of connective tissue that determines its properties?
[remember] 11/1000: What is the name given to an increase in tissue fluid volume?
[remember] 12/1000: What is the normal size range of lymph nodes?
[remember] 13/1000: From which embryonic tissue do connective tissue ce

Split them into 3 lists.

In [ ]:
remember_questions  = [q for q, lbl in all_questions if lbl == "remember"]
understand_questions = [q for q, lbl in all_questions if lbl == "understand"]
apply_questions     = [q for q, lbl in all_questions if lbl == "apply"]

## Accuracy of Single Agent

Use our classifier to compute accuracy for remember questions.

In [ ]:
_, gpt_labels = classify_questions(remember_questions)
true_labels = ["remember"] * len(remember_questions)

remember_accuracy = compute_accuracy(true_labels, gpt_labels)
print(f"Remember Accuracy: {remember_accuracy:.2%}")

Accuracy: 100.00%
Remember Accuracy: 100.00%


Inspect any misclasffications.

In [ ]:
df = pd.DataFrame({
    "Question": remember_questions,
    "GPT Label": gpt_labels
})

print(df.to_string(index=False))

                                                                                              Question  GPT Label
                                                                 What are the four basic tissue types?   remember
                                         What determines the physical properties of connective tissue? understand
                                     What term is used to describe an increase in tissue fluid volume?   remember
                                                            What size do lymph nodes normally measure?   remember
                                      What embryonic tissue do connective tissue cells originate from?   remember
What type of connective tissue has parallel collagen fibres providing tensile strength along one axis?   remember
                         What type of cells are embedded within the extracellular matrix of cartilage?   remember
                                  What type of bone houses red bone marrow, the site of 

Use classifieer to find accruacy of understand questions.

In [ ]:
_, gpt_labels = classify_questions(understand_questions)
true_labels = ["understand"] * len(understand_questions)

understand_accuracy = compute_accuracy(true_labels, gpt_labels)
print(f"Understand Accuracy: {understand_accuracy:.2%}")

Accuracy: 100.00%
Understand Accuracy: 100.00%


Use classifer to find accuracy of apply question.

In [ ]:
_, gpt_labels = classify_questions(apply_questions)
true_labels = ["apply"] * len(apply_questions)

apply_accuracy = compute_accuracy(true_labels, gpt_labels)
print(f"Apply Accuracy: {apply_accuracy:.2%}")

Accuracy: 100.00%
Apply Accuracy: 100.00%


Inspect any misclassifications.

In [ ]:
df = pd.DataFrame({
    "Question": apply_questions,
    "GPT Label": gpt_labels
})

print(df.to_string(index=False))

                                                                                                                                                                                             Question  GPT Label
                                                                         If neural tissue in an organ is replaced with connective tissue, what will be the resulting effect on that organ’s function?      apply
                                                If a connective tissue’s cells stopped producing the organic molecules that bind tissue fluid, what would most likely happen to its ground substance?      apply
                                                          If the adhesive glycoproteins in connective tissue were absent, what would immediately happen to the stability of the extracellular matrix?      apply
                                                              If a patient experiences a blockage of lymphatic drainage from the axillae, what would likely happen t

Both single and multiple agents perform well, showing that a single agent is sufficient.